<a href="https://colab.research.google.com/github/jumanazez/DSproject/blob/main/Phase3_Machine_Learning_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# **Phase 3: Data Preparation and Baseline Model**

## Thmanyah Podcast Engagement Analysis

This notebook covers data preparation and baseline modelling for Phase 3 of the IT362 Data Science project.

The main objective is to establish a baseline regression model for predicting episode views using podcast characteristics, including episode topic, duration, and publication information.

The baseline results will serve as a reference for evaluating more advanced regression models.



## 1. Loading the Dataset

The cleaned Thmanyah podcast dataset from Phase 2 is loaded for modelling. The dataset contains 698 podcast episodes and 17 columns.

The dataset dimensions and first five records are displayed to verify that the data has been loaded correctly.

In [2]:

import pandas as pd
import numpy as np

df = pd.read_csv("/content/thmanyah_cleaned_dataset.csv")

print(df.shape)
display(df.head())


(698, 17)


,video_id,podcast,published_at,title,description,category_id,duration,views,likes,comments,duration_minutes,year,month,day_of_week,engagement_rate,clean_title,topic
0,HZKdvcoyzwI,Swalif Business,2026-01-19 12:00:18+00:00,كيف تضمن الشركات استدامتها | بودكاست سوالف بزنس,هناك أكثر من مليون منشأة فعّالة في السعودية، م...,24,PT1H28M55S,135283,1729,53,88.916667,2026,1,Monday,1.317239,كيف تضمن الشركات استدامتها,Business & Finance
1,DA0oZJ8Ly-8,Swalif Business,2025-11-24 15:02:36+00:00,كيف تستفيد من الذكاء الاصطناعي في شركتك | بودك...,وسط موجة المبالغة والترويج المفرط للذكاء الاصط...,24,PT2H5M55S,98130,1220,176,125.916667,2025,11,Monday,1.422603,كيف تستفيد من الذكاء الاصطناعي في شركتك,Technology
2,YI5PVi26Ixg,Swalif Business,2025-11-18 13:01:09+00:00,كيف تعمل شركات تصنيع المكياج | بودكاست سوالف بزنس,برزت في السنوات الأخيرة علامات تجارية في عالم ...,24,PT1H19M42S,39131,508,112,79.700000,2025,11,Tuesday,1.584422,كيف تعمل شركات تصنيع المكياج,Other
3,t7IrlGQ3aPM,Swalif Business,2025-11-10 15:01:42+00:00,أين الفرص الاستثمارية في قطاع الصناعات العسكري...,نستضيف في هذه الحلقة بدر المصيبيح، المدير العا...,24,PT2H21M58S,90831,1227,170,141.966667,2025,11,Monday,1.538021,أين الفرص الاستثمارية في قطاع الصناعات العسكرية,Business & Finance
4,JzQJahCyZcw,Swalif Business,2025-11-03 19:36:00+00:00,كيف نجح تطبيق توصيل سوري رغم الحرب والعقوبات |...,هذه الحلقة مختلفة تمامًا عن الحلقات السابقة، م...,24,PT1H35M29S,35332,543,199,95.483333,2025,11,Monday,2.100079,كيف نجح تطبيق توصيل سوري رغم الحرب والعقوبات,Technology


## 2. Feature and Target Selection

The target variable is `views`, representing the number of views received by each podcast episode.

Six input features are selected: `podcast`, `topic`, `duration_minutes`, `year`, `month`, and `day_of_week`.

The topic feature is particularly important because the project's main research question investigates differences in audience reach and engagement across podcast topics.

Likes, comments, and engagement rate are excluded because they represent audience interactions measured after publication.

In [3]:

features = [
    "podcast",
    "topic",
    "duration_minutes",
    "year",
    "month",
    "day_of_week"
]

X = df[features]
y = df["views"]


## 3. Train-Test Split

The dataset is randomly divided into 80% training data and 20% testing data using `random_state=42` for reproducibility.

The training set contains 558 episodes, while the testing set contains 140 episodes.

This split allows the models to be evaluated on data that was not used during training.

In [4]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)


Training: (558, 6)
Testing: (140, 6)


## 4. Preparing Features Without Topic

An additional feature set is created by excluding the `topic` variable.

This will allow later models to be evaluated both with and without topic information, helping assess whether episode topics improve the prediction of views.

The same training and testing observations are maintained to ensure a fair comparison.

In [5]:

# Features without the topic column
features_without_topic = [
    "podcast",
    "duration_minutes",
    "year",
    "month",
    "day_of_week"
]

X_train_no_topic = X_train[features_without_topic]
X_test_no_topic = X_test[features_without_topic]


## 5. Categorical Feature Encoding

The selected features include both categorical and numerical variables.

One-Hot Encoding is used to transform categorical features (`podcast`, `topic`, and `day_of_week`) into numerical representations.

Numerical features (`duration_minutes`, `year`, and `month`) are passed through without modification.

A ColumnTransformer is used to define these preprocessing steps consistently.

In [6]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = [
    "podcast", "topic", "day_of_week"
]

numeric_features = [
    "duration_minutes", "year", "month"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)


## 6. Baseline Model Development

A Dummy Regressor with the median strategy is used as the baseline model.

This model predicts the median number of views from the training dataset for all test observations, without learning relationships between the input features and the target variable.

The median strategy was selected because the exploratory data analysis in Phase 2 revealed a highly right-skewed distribution of views, with some episodes receiving exceptionally high view counts.

The baseline model provides a reference for evaluating more advanced regression models.

In [7]:

from sklearn.dummy import DummyRegressor
from sklearn.pipeline import Pipeline

baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DummyRegressor(strategy="median"))
])

baseline_model.fit(X_train, y_train)

y_pred = baseline_model.predict(X_test)

print("Baseline model trained successfully!")


Baseline model trained successfully!


## 7. Baseline Model Evaluation

The baseline model is evaluated using three regression metrics:

- **Mean Absolute Error (MAE):** Measures the average absolute difference between actual and predicted views.
- **Root Mean Squared Error (RMSE):** Measures prediction error while giving greater weight to large errors.
- **R-squared (R²):** Measures how well the predictions explain variation in the target compared with predicting its test-set mean.

These metrics will be used to compare the baseline model with the advanced regression models.

In [8]:

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Baseline Model Results:")
print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2:", round(r2, 4))


Baseline Model Results:
MAE: 1568733.84
RMSE: 13794876.21
R2: -0.012


### Baseline Evaluation Results and Interpretation

The baseline model achieved the following results:

- **MAE:** 1,568,733.84
- **RMSE:** 13,794,876.21
- **R²:** -0.012

The relatively high MAE indicates substantial prediction errors. The RMSE is considerably higher than the MAE, suggesting that some episodes have particularly large prediction errors, consistent with the extreme view counts observed during EDA.

The negative R² indicates that the baseline performed slightly worse than predicting the mean of the test-set views.

These results are expected from a simple baseline model and will serve as a benchmark for evaluating more advanced regression models.

In [9]:

preprocessor_no_topic = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         ["podcast", "day_of_week"]),
        ("num", "passthrough",
         ["duration_minutes", "year", "month"])
    ]
)


## 8. Saving Training and Testing Datasets

The training and testing datasets are saved as separate CSV files.

Each file includes the selected input features and the target variable (`views`).

These files will be used to train and evaluate the additional regression models using the same data split, ensuring consistent and fair comparisons.

In [10]:

train_data = X_train.copy()
train_data["views"] = y_train

test_data = X_test.copy()
test_data["views"] = y_test

train_data.to_csv("phase3_train.csv", index=False)
test_data.to_csv("phase3_test.csv", index=False)

print("Files saved successfully!")


Files saved successfully!


## 2. Machine Learning Models

In this section, two regression models, Linear Regression and Random Forest Regressor, are developed using the same training and testing datasets and preprocessing configuration established in the baseline model.

## Linear Regression Model

Linear Regression was selected as the first predictive model because it provides a simple and interpretable approach to estimating episode views from podcast characteristics.

The model uses the same training and testing sets and preprocessing configuration as the baseline model to ensure a consistent comparison.

In [11]:

from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

# Build Linear Regression using the same preprocessing
linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

# Train the model
linear_model.fit(X_train, y_train)

# Predict views for the test set
linear_predictions = linear_model.predict(X_test)

print("Linear Regression trained successfully!")
print("Number of predictions:", len(linear_predictions))

print("\nFirst 5 predictions:")
print(linear_predictions[:5])



Linear Regression trained successfully!
Number of predictions: 140

First 5 predictions:
[ 474923.60201904 1488904.09407943  216809.15553164  505470.62769607
 -259032.54888546]


## Random Forest Regressor Model

Random Forest Regressor was selected as the second predictive model because it can capture complex, non-linear relationships between podcast characteristics and episode views.

Unlike Linear Regression, Random Forest combines predictions from multiple decision trees, allowing it to model more flexible patterns in the data.

The model uses the same training and testing datasets and preprocessing configuration to ensure a fair comparison with Linear Regression and the baseline model.

In [12]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

# Build Random Forest using the same preprocessing
random_forest_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

# Train the model
random_forest_model.fit(X_train, y_train)

# Predict views for the test set
rf_predictions = random_forest_model.predict(X_test)

print("Random Forest trained successfully!")
print("Number of predictions:", len(rf_predictions))

print("\nFirst 5 predictions:")
print(rf_predictions[:5])


Random Forest trained successfully!
Number of predictions: 140

First 5 predictions:
[ 208394.4  2059131.27  106417.92  145186.91  174319.8 ]


## Prediction Validation

After training both regression models, their predictions were checked for invalid negative values. Since episode views cannot be negative, this check helps identify limitations in the predictions before model evaluation.

Both models were trained and tested using the same data split to ensure consistency.

In [13]:

import numpy as np

print("Linear Regression:")
print("Negative predictions:", np.sum(linear_predictions < 0))
print("Minimum prediction:", round(linear_predictions.min(), 2))
print("Maximum prediction:", round(linear_predictions.max(), 2))

print("\nRandom Forest:")
print("Negative predictions:", np.sum(rf_predictions < 0))
print("Minimum prediction:", round(rf_predictions.min(), 2))
print("Maximum prediction:", round(rf_predictions.max(), 2))


Linear Regression:
Negative predictions: 41
Minimum prediction: -942777.76
Maximum prediction: 2486015.08

Random Forest:
Negative predictions: 0
Minimum prediction: 57855.53
Maximum prediction: 21524376.38


### Prediction Validation Findings

Both models successfully generated predictions for all **140 test episodes**.

However, Linear Regression produced **41 negative predictions (29.3%)**, with a minimum predicted value of **−942,777.76 views**. Since view counts cannot be negative, this highlights a limitation of the linear model when predicting highly skewed audience reach data.

In contrast, Random Forest produced **no negative predictions**, with predicted views ranging from **57,855.53 to 21,524,376.38**.

Although Random Forest generated more realistic non-negative predictions, **model accuracy cannot be determined from this check alone**. The final comparison will use MAE, RMSE, and R².

## Model Development Summary

Two regression models were successfully developed using the same training and testing datasets and preprocessing configuration as the baseline model.

Linear Regression was used to examine linear relationships between episode characteristics and views, while Random Forest was used to capture more complex, non-linear patterns.

The trained models and their predictions are ready for the model evaluation stage.

In [17]:

# Save actual and predicted views for model evaluation

predictions_df = pd.DataFrame({
    "Actual_Views": y_test.to_numpy(),
    "Linear_Regression_Predictions": linear_predictions,
    "Random_Forest_Predictions": rf_predictions
})

predictions_df.to_csv(
    "/content/phase3_model_predictions.csv",
    index=False
)

print("Predictions saved successfully!")
print("Shape:", predictions_df.shape)

display(predictions_df.head())


Predictions saved successfully!
Shape: (140, 3)


,Actual_Views,Linear_Regression_Predictions,Random_Forest_Predictions
0,320583,4.749236e+05,208394.40
1,114521,1.488904e+06,2059131.27
2,77259,2.168092e+05,106417.92
3,231003,5.054706e+05,145186.91
4,71534,-2.590325e+05,174319.80


In [18]:

from google.colab import files

files.download("/content/phase3_model_predictions.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>